In [ ]:
from pyspark.sql.functions import col, when, lit, concat, lower
from pyspark.sql.types import StructType, StructField, StringType

In [ ]:
dbutils.widgets.dropdown(
    name="segment",
    defaultValue="TD",
    choices=["BAILS", "TD", "SBAILS", "JOH", "FTA", "FPA", "UTA"],
    label="Segment"
)
segment = dbutils.widgets.get("segment")
print(f"Selected segment: {segment}")

In [ ]:
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key   = config.first()["lz_key"].strip().lower()
print(f"env_name={env_name}, lz_key={lz_key}")

keyvault_name = f"ingest{lz_key}-meta002-{env_name}"
print(f"keyvault_name={keyvault_name}")

In [ ]:
try:
    client_secret = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-SECRET')
    tenant_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-TENANT-ID')
    client_id = dbutils.secrets.get(scope=keyvault_name, key='SERVICE-PRINCIPLE-CLIENT-ID')
    print("Successfully retrieved all Service Principal secrets from Key Vault")
except Exception as e:
    print(f"Failed to retrieve secrets: {e}", exc_info=True)
    raise

In [ ]:
curated_storage_account = f"ingest{lz_key}curated{env_name}"

for storage_account in [curated_storage_account]:
    try:
        configs = {
            f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
            f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
                "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
            f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
            f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
            f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
                f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
        }
        for key, val in configs.items():
            spark.conf.set(key, val)
        print(f"Configured OAuth for {storage_account}")
    except Exception as e:
        print(f"Failed to configure OAuth for {storage_account}: {e}", exc_info=True)
        raise

In [ ]:
silver_base = f"abfss://silver@{curated_storage_account}.dfs.core.windows.net"
gold_base   = f"abfss://gold@{curated_storage_account}.dfs.core.windows.net"
blob_host   = f"https://ingest{lz_key}curated{env_name}.blob.core.windows.net/gold"

seg_lower = segment.lower()
APPEALS_SEGMENTS = {"FTA", "FPA", "UTA"}
ABBREV = {
    "BAILS":  "bl",
    "TD":     "td",
    "SBAILS": "sbl",
    "JOH":    "joh",
    "FTA":    "aplfta",
    "FPA":    "aplfpa",
    "UTA":    "apluta",
}

if segment not in ABBREV:
    raise ValueError(f"Unknown segment '{segment}'. Valid options: {list(ABBREV)}")

abbrev = ABBREV[segment]

if segment in APPEALS_SEGMENTS:
    seg_path = f"APPEALS/ARIA{segment}"
    cfg = {
        "pub_audit_path": f"{silver_base}/ARIADM/ARM/AUDIT/{seg_path}/apl_{seg_lower}_pub_audit_table",
        "ack_audit_path": f"{silver_base}/ARIADM/ARM/AUDIT/APPEALS/{seg_lower}_ack_audit_table",
        "blob_base_url":  f"{blob_host}/ARIADM/ARM/{seg_path}",
        "topic":          f"evh-{abbrev}-pub-{lz_key}-uks-dlrm-01",
    }
else:
    cfg = {
        "pub_audit_path": f"{silver_base}/ARIADM/ARM/AUDIT/{segment}/{abbrev}_pub_audit_table",
        "ack_audit_path": f"{silver_base}/ARIADM/ARM/AUDIT/{segment}/{abbrev}_ack_audit_table",
        "blob_base_url":  f"{blob_host}/ARIADM/ARM/{segment}",
        "topic":          f"evh-{abbrev}-pub-{lz_key}-uks-dlrm-01",
    }

print(f"pub_audit_path : {cfg['pub_audit_path']}")
print(f"ack_audit_path : {cfg['ack_audit_path']}")
print(f"topic          : {cfg['topic']}")

## Audit Comparison

In [ ]:
pub_df = spark.read.format("delta").load(cfg["pub_audit_path"])
published_files = (
    pub_df
    .filter(col("status") == "success")
    .select(col("file_name").alias("filename"))
    .distinct()
)
published_count = published_files.count()
print(f"Successfully published files: {published_count}")

In [ ]:
ack_df = spark.read.format("delta").load(cfg["ack_audit_path"])
acknowledged_files = ack_df.select("filename").distinct()
acknowledged_count = acknowledged_files.count()
print(f"Acknowledged files: {acknowledged_count}")

In [ ]:
missing_df = published_files.join(acknowledged_files, on="filename", how="left_anti").cache()
missing_count = missing_df.count()
print(f"Published but not acknowledged: {missing_count}")

In [ ]:
missing_df.display()

In [ ]:
if missing_count == 0:
    print("No missing events to republish. Exiting.")
    dbutils.notebook.exit("Nothing to republish — all published files have been acknowledged")

## Republish Missing Events

In [ ]:
eh_kv_secret = dbutils.secrets.get(scope=keyvault_name, key="RootManageSharedAccessKey")
eventhubs_hostname = f"ingest{lz_key}-integration-eventHubNamespace001-{env_name}.servicebus.windows.net:9093"

kafka_conf = {
    'bootstrap.servers': eventhubs_hostname,
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': '$ConnectionString',
    'sasl.password': eh_kv_secret,
    'retries': 5,
    'enable.idempotence': True
}
broadcast_conf = sc.broadcast(kafka_conf)

topic = cfg["topic"]
print(f"Event Hub topic: {topic}")

In [ ]:
blob_base_url = cfg["blob_base_url"]

missing_with_url = (
    missing_df
    .withColumn(
        "suffix",
        when(lower(col("filename")).endswith(".html"), lit("HTML"))
        .when(lower(col("filename")).endswith(".json"), lit("JSON"))
        .when(lower(col("filename")).endswith(".a360"), lit("A360"))
    )
    .withColumn(
        "blob_url",
        concat(lit(blob_base_url + "/"), col("suffix"), lit("/"), col("filename"))
    )
    .withColumnRenamed("filename", "file_path")
)

unmatched = missing_with_url.filter(col("suffix").isNull()).count()
if unmatched > 0:
    print(f"WARNING: {unmatched} files have an unrecognised extension and will be skipped")

missing_with_url = missing_with_url.filter(col("suffix").isNotNull())
missing_with_url.display()

In [ ]:
num_spark_partitions = 8
optimized_df = missing_with_url.repartition(num_spark_partitions)


def process_partition(partition):
    import logging
    from confluent_kafka import Producer
    from datetime import datetime

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger('KafkaProducer')

    failure_list = []
    success_list = []

    producer = Producer(**broadcast_conf.value)

    def delivery_report(err, msg):
        key_str = msg.key().decode('utf-8') if msg.key() is not None else "Unknown"
        if err is not None:
            print(f"Delivery failed for {key_str}: {err}")
            failure_list.append((key_str, "failure", str(err), datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")))
        else:
            success_list.append((key_str, "success", "", datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")))

    def produce_message(key, value):
        try:
            producer.produce(topic=topic, key=key, value=value, callback=delivery_report)
        except BufferError:
            raise
        except Exception as e:
            print(f"Unexpected error during produce: {e}")
            failure_list.append((key.decode('utf-8'), "failure", str(e), datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")))

    for row in partition:
        producer.poll(0)

        if row.file_path is None or row.blob_url is None:
            print(f"Skipping row with missing file_path/blob_url: {row}")
            failure_list.append((row.file_path, "failure", "Missing file_path or blob_url", datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")))
            continue

        key_bytes = row.file_path.encode('utf-8')
        value_bytes = row.blob_url.encode('utf-8')

        try:
            produce_message(key_bytes, value_bytes)
        except BufferError:
            print("Buffer full. Polling for 10s")
            producer.poll(10)
            try:
                produce_message(key_bytes, value_bytes)
            except BufferError:
                print("Buffer still full. Polling for 30s")
                producer.poll(30)
                try:
                    produce_message(key_bytes, value_bytes)
                except BufferError as e:
                    failure_list.append((row.file_path, "failure", str(e), datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")))

    try:
        producer.flush()
    except Exception as e:
        print(f"Flush error: {e}")

    return success_list + failure_list


result_schema = StructType([
    StructField("file_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
    StructField("timestamp", StringType(), True),
])

result_rdd = optimized_df.rdd.mapPartitions(process_partition)
result_df = spark.createDataFrame(result_rdd, result_schema).cache()
result_df.count()

display(result_df)

In [ ]:
failed_df = result_df.filter(col("status") == "failure")
display(failed_df)
print(f"Failed: {failed_df.count()}")

In [ ]:
successful = result_df.filter(col("status") == "success").count()
failed     = result_df.filter(col("status") == "failure").count()

print(f"Segment          : {segment}")
print(f"Missing events   : {missing_count}")
print(f"Republish success: {successful}")
print(f"Republish failed : {failed}")

dbutils.notebook.exit({"segment": segment, "missing": missing_count, "republish_success": successful, "republish_failed": failed})